In [ ]:
# Mapeia diretório de uso
from google.colab import drive
import os
import pandas as pd
import numpy as np

# Se você estiver usando o Google Colab, descomente as duas linhas abaixo
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/MyDrive/TCC_2/dados')

In [ ]:
# ============================================================
# DOWNLOAD, FILTRO E VALIDAÇÃO RAIS - BRASIL
#
# Anos:
#   2020, 2021, 2022, 2023, 2024 e 2025
#
# ATENÇÃO:
#   2022 será incluído provisoriamente para auditoria.
#   A decisão de mantê-lo ou excluí-lo será tomada depois
#   da análise das variáveis de afastamento.
#
# População:
#   Professores do Ensino Fundamental e Ensino Médio
#
# Famílias CBO:
#   2312 = Professores de nível superior do Ensino Fundamental
#          - anos iniciais
#   2313 = Professores de nível superior do Ensino Fundamental
#          - anos finais
#   2321 = Professores do Ensino Médio
#   3312 = Professores de nível médio no Ensino Fundamental
#   3321 = Professores leigos no Ensino Fundamental
#
# Estratégia:
#   1. reutiliza arquivos .7z já existentes no Google Drive
#   2. baixa apenas os arquivos ausentes
#   3. descompacta temporariamente no disco do Colab
#   4. lê os arquivos em blocos
#   5. mantém somente as 5 famílias CBO selecionadas
#   6. cria UF a partir de Município / Município - Código
#   7. valida UF contra o grupo geográfico do arquivo
#   8. grava Parquet filtrado no Google Drive
#   9. remove o arquivo bruto temporário
#
# Os Parquets anteriores NÃO são sobrescritos.
#
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive
from ftplib import FTP
from urllib.parse import quote

import os
import re
import shutil
import subprocess
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 3. CONFIGURAÇÕES GERAIS
# ============================================================

# ------------------------------------------------------------
# Arquivos compactados .7z
# Já existentes serão reutilizados.
# ------------------------------------------------------------

PASTA_BASE = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_OUTROS_ESTADOS'
)


# ------------------------------------------------------------
# NOVA pasta para os Parquets corrigidos
#
# Não usar RAIS_PROFESSORES, pois aquela pasta contém
# o primeiro filtro com apenas 3 famílias CBO.
# ------------------------------------------------------------

PASTA_FILTRADA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_PROFESSORES_5CBO'
)


# ------------------------------------------------------------
# Pasta temporária no próprio Colab
# ------------------------------------------------------------

PASTA_TEMP = (
    '/content/RAIS_TEMP'
)


# ------------------------------------------------------------
# FTP oficial
# ------------------------------------------------------------

HOST = 'ftp.mtps.gov.br'

BASE_FTP = (
    '/pdet/microdados/RAIS'
)


# ============================================================
# 4. ANOS
# ============================================================

ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]


# ============================================================
# 5. GRUPOS REGIONAIS DA RAIS
# ============================================================

GRUPOS = {

    'NORTE':
        'RAIS_VINC_PUB_NORTE',

    'NORDESTE':
        'RAIS_VINC_PUB_NORDESTE',

    'CENTRO_OESTE':
        'RAIS_VINC_PUB_CENTRO_OESTE',

    'MG_ES_RJ':
        'RAIS_VINC_PUB_MG_ES_RJ',

    'SP':
        'RAIS_VINC_PUB_SP',

    'SUL':
        'RAIS_VINC_PUB_SUL'
}


# ============================================================
# 6. FAMÍLIAS CBO
# ============================================================

FAMILIAS_CBO = {

    '2312':
        'Ensino Fundamental - anos iniciais - nível superior',

    '2313':
        'Ensino Fundamental - anos finais - nível superior',

    '2321':
        'Ensino Médio',

    '3312':
        'Ensino Fundamental - nível médio',

    '3321':
        'Professor leigo - Ensino Fundamental'
}


# ============================================================
# 7. CÓDIGOS DE UF
# ============================================================

MAPA_UF = {

    '11': 'RO',
    '12': 'AC',
    '13': 'AM',
    '14': 'RR',
    '15': 'PA',
    '16': 'AP',
    '17': 'TO',

    '21': 'MA',
    '22': 'PI',
    '23': 'CE',
    '24': 'RN',
    '25': 'PB',
    '26': 'PE',
    '27': 'AL',
    '28': 'SE',
    '29': 'BA',

    '31': 'MG',
    '32': 'ES',
    '33': 'RJ',
    '35': 'SP',

    '41': 'PR',
    '42': 'SC',
    '43': 'RS',

    '50': 'MS',
    '51': 'MT',
    '52': 'GO',
    '53': 'DF'
}


# ============================================================
# 8. UFs ESPERADAS POR GRUPO
# ============================================================

UFS_ESPERADAS = {

    'NORTE': {
        'AC',
        'AP',
        'AM',
        'PA',
        'RO',
        'RR',
        'TO'
    },

    'NORDESTE': {
        'AL',
        'BA',
        'CE',
        'MA',
        'PB',
        'PE',
        'PI',
        'RN',
        'SE'
    },

    'CENTRO_OESTE': {
        'DF',
        'GO',
        'MT',
        'MS'
    },

    'MG_ES_RJ': {
        'MG',
        'ES',
        'RJ'
    },

    'SP': {
        'SP'
    },

    'SUL': {
        'PR',
        'SC',
        'RS'
    }
}


# ============================================================
# 9. TAMANHO DOS BLOCOS
# ============================================================

CHUNKSIZE = 300_000


# ============================================================
# 10. CRIAR PASTAS
# ============================================================

os.makedirs(
    PASTA_BASE,
    exist_ok=True
)

os.makedirs(
    PASTA_FILTRADA,
    exist_ok=True
)

os.makedirs(
    PASTA_TEMP,
    exist_ok=True
)


# ============================================================
# 11. INSTALAR 7ZIP
# ============================================================

subprocess.run(

    [
        'apt-get',
        'update'
    ],

    stdout=subprocess.DEVNULL,

    check=True
)


subprocess.run(

    [
        'apt-get',
        'install',
        '-y',
        'p7zip-full'
    ],

    stdout=subprocess.DEVNULL,

    check=True
)


# ============================================================
# 12. NORMALIZAR TEXTO
# ============================================================

def normalizar_texto(texto):

    texto = str(
        texto
    )

    texto = unicodedata.normalize(
        'NFKD',
        texto
    )

    texto = ''.join(

        caractere

        for caractere in texto

        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r'[^A-Z0-9]+',
        ' ',
        texto
    )

    texto = re.sub(
        r'\s+',
        ' ',
        texto
    ).strip()

    return texto


# ============================================================
# 13. LISTAR ARQUIVOS DO FTP
# ============================================================

def listar_ftp(ano):

    erros = []


    for encoding in [

        'utf-8',
        'cp1252',
        'latin1'

    ]:

        ftp = None

        try:

            ftp = FTP(

                HOST,

                timeout=180,

                encoding=encoding
            )


            ftp.login()


            ftp.cwd(
                f'{BASE_FTP}/{ano}'
            )


            arquivos = ftp.nlst()


            ftp.quit()


            return arquivos


        except Exception as erro:

            erros.append(
                f'{encoding}: {erro}'
            )


            try:

                if ftp:

                    ftp.close()

            except Exception:

                pass


    raise RuntimeError(

        f'Não foi possível listar '
        f'os arquivos do ano {ano}:\n'

        +
        '\n'.join(erros)
    )


# ============================================================
# 14. LOCALIZAR ARQUIVO NO FTP
# ============================================================

def localizar_arquivo(
    arquivos,
    nome_base
):

    candidatos = []


    for arquivo in arquivos:

        nome = (

            os.path.basename(
                arquivo
            )

            .upper()
        )


        if (

            nome_base.upper()
            in nome

            and

            nome.endswith(
                '.7Z'
            )
        ):

            candidatos.append(
                arquivo
            )


    if len(
        candidatos
    ) != 1:

        raise RuntimeError(

            f'Não foi encontrado exatamente '
            f'um arquivo para:\n'

            f'{nome_base}\n\n'

            f'Candidatos encontrados:\n'

            f'{candidatos}'
        )


    return candidatos[0]


# ============================================================
# 15. IDENTIFICAR LAYOUT
# ============================================================

def identificar_layout(
    caminho
):

    separadores = [
        ';',
        ',',
        '\t',
        '|'
    ]


    encodings = [
        'utf-8',
        'cp1252',
        'latin1'
    ]


    for encoding in encodings:

        for separador in separadores:

            try:

                cabecalho = pd.read_csv(

                    caminho,

                    sep=separador,

                    encoding=encoding,

                    nrows=0
                )


                colunas = list(
                    cabecalho.columns
                )


                # --------------------------------------------
                # Precisa haver várias colunas
                # --------------------------------------------

                if len(
                    colunas
                ) < 5:

                    continue


                # ============================================
                # IDENTIFICAR CBO
                # ============================================

                cbo_coluna = None


                for coluna in colunas:

                    nome = normalizar_texto(
                        coluna
                    )


                    if (

                        'CBO' in nome

                        and

                        '2002' in nome

                        and

                        'OCUP' in nome
                    ):

                        cbo_coluna = coluna

                        break


                if cbo_coluna is None:

                    continue


                # ============================================
                # IDENTIFICAR MUNICÍPIO
                #
                # IMPORTANTE:
                #
                # Queremos:
                #
                # 2020-2022:
                #   Município
                #
                # 2023-2025:
                #   Município - Código
                #
                # NÃO queremos:
                #
                #   Mun Trab
                #   Município Trab - Código
                #
                # ============================================

                municipio_coluna = None


                # --------------------------------------------
                # 1ª tentativa:
                # nomes exatos normalizados
                # --------------------------------------------

                for coluna in colunas:

                    nome = normalizar_texto(
                        coluna
                    )


                    if nome in {

                        'MUNICIPIO',

                        'MUNICIPIO CODIGO'

                    }:

                        municipio_coluna = (
                            coluna
                        )

                        break


                # --------------------------------------------
                # 2ª tentativa:
                # procurar Município,
                # excluindo explicitamente TRAB
                # --------------------------------------------

                if municipio_coluna is None:

                    for coluna in colunas:

                        nome = (
                            normalizar_texto(
                                coluna
                            )
                        )


                        if (

                            'MUNICIPIO'
                            in nome

                            and

                            'TRAB'
                            not in nome
                        ):

                            municipio_coluna = (
                                coluna
                            )

                            break


                if municipio_coluna is None:

                    raise RuntimeError(

                        'Coluna de município '
                        'não localizada.\n'

                        f'Colunas disponíveis:\n'
                        f'{colunas}'
                    )


                # ============================================
                # RETORNO
                # ============================================

                return {

                    'encoding':
                        encoding,

                    'sep':
                        separador,

                    'cbo':
                        cbo_coluna,

                    'municipio':
                        municipio_coluna,

                    'colunas':
                        colunas
                }


            except RuntimeError:

                raise


            except Exception:

                continue


    raise RuntimeError(

        f'Não foi possível identificar '
        f'o layout de:\n'

        f'{caminho}'
    )


# ============================================================
# 16. FUNÇÃO PARA EXTRAIR CÓDIGO DE MUNICÍPIO
# ============================================================

def extrair_codigo_municipio(
    serie
):

    # --------------------------------------------------------
    # Aceita 6 ou 7 dígitos.
    #
    # Exemplos encontrados nos microdados:
    #
    # 150140
    # 355030
    #
    # Se eventualmente aparecer código com 7 dígitos,
    # ele também será aceito.
    # --------------------------------------------------------

    codigo = (

        serie

        .astype(
            'string'
        )

        .str.strip()

        .str.extract(
            r'(\d{6,7})',
            expand=False
        )
    )


    return codigo


# ============================================================
# 17. FILTRAR PROFESSORES
# ============================================================

def filtrar_professores(

    caminho_arquivo,

    ano,

    grupo,

    saida_parquet
):

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    layout = identificar_layout(
        caminho_arquivo
    )


    print(
        f'Encoding: '
        f'{layout["encoding"]}'
    )

    print(
        f'Separador: '
        f'{repr(layout["sep"])}'
    )

    print(
        f'CBO: '
        f'{layout["cbo"]}'
    )

    print(
        f'Município utilizado: '
        f'{layout["municipio"]}'
    )


    # --------------------------------------------------------
    # Garantia adicional:
    # jamais usar Município de Trabalho
    # --------------------------------------------------------

    nome_municipio_normalizado = (
        normalizar_texto(
            layout['municipio']
        )
    )


    if (
        'TRAB'
        in nome_municipio_normalizado
    ):

        raise RuntimeError(

            'ERRO DE SEGURANÇA: '
            'foi selecionada uma coluna '
            'de Município de Trabalho:\n'

            f'{layout["municipio"]}'
        )


    # --------------------------------------------------------
    # Contadores
    # --------------------------------------------------------

    total_linhas = 0

    total_professores = 0

    total_uf_nula = 0

    contagem_familias = {}

    contagem_ufs = {}

    writer = None


    try:

        leitor = pd.read_csv(

            caminho_arquivo,

            sep=layout[
                'sep'
            ],

            encoding=layout[
                'encoding'
            ],

            dtype=str,

            chunksize=CHUNKSIZE,

            low_memory=False,

            on_bad_lines='error'
        )


        # ====================================================
        # LOOP EM CHUNKS
        # ====================================================

        for numero_chunk, chunk in enumerate(

            leitor,

            start=1
        ):

            total_linhas += len(
                chunk
            )


            # ================================================
            # CBO
            # ================================================

            cbo = (

                chunk[
                    layout['cbo']
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{6})',
                    expand=False
                )
            )


            familia = (
                cbo.str[:4]
            )


            mascara = (
                familia.isin(
                    FAMILIAS_CBO.keys()
                )
            )


            prof = (

                chunk.loc[
                    mascara
                ]

                .copy()
            )


            # --------------------------------------------
            # Nenhum professor nesse bloco
            # --------------------------------------------

            if len(
                prof
            ) == 0:

                print(

                    f'Chunk {numero_chunk}: '

                    f'{len(chunk):,} linhas | '

                    f'0 professores'
                )

                continue


            # ================================================
            # VARIÁVEIS PADRONIZADAS
            # ================================================

            prof[
                'CBO_padronizada'
            ] = (

                cbo.loc[
                    mascara
                ]

                .values
            )


            prof[
                'Familia_CBO'
            ] = (

                familia.loc[
                    mascara
                ]

                .values
            )


            prof[
                'Nivel_Ensino'
            ] = (

                prof[
                    'Familia_CBO'
                ]

                .map(
                    FAMILIAS_CBO
                )
            )


            prof[
                'Ano'
            ] = str(
                ano
            )


            prof[
                'Grupo_Origem'
            ] = grupo


            # ================================================
            # MUNICÍPIO
            # ================================================

            municipio = (

                extrair_codigo_municipio(

                    prof[
                        layout[
                            'municipio'
                        ]
                    ]
                )
            )


            # --------------------------------------------
            # Preservar código padronizado
            # --------------------------------------------

            prof[
                'Municipio_padronizado'
            ] = municipio


            # ================================================
            # UF
            # ================================================

            codigo_uf = (
                municipio.str[:2]
            )


            prof[
                'UF'
            ] = (

                codigo_uf

                .map(
                    MAPA_UF
                )
            )


            # ================================================
            # VALIDAR UFs
            # ================================================

            ufs_encontradas_chunk = set(

                prof[
                    'UF'
                ]

                .dropna()

                .unique()
            )


            ufs_invalidas = (

                ufs_encontradas_chunk

                -

                UFS_ESPERADAS[
                    grupo
                ]
            )


            if len(
                ufs_invalidas
            ) > 0:

                raise RuntimeError(

                    f'UFs incompatíveis '
                    f'com o grupo {grupo}: '

                    f'{sorted(ufs_invalidas)}'
                )


            # ================================================
            # CONTAGEM DE UFs NULAS
            # ================================================

            uf_nula_chunk = (

                prof[
                    'UF'
                ]

                .isna()

                .sum()
            )


            total_uf_nula += int(
                uf_nula_chunk
            )


            # ================================================
            # CONTAGEM POR UF
            # ================================================

            contagem_chunk_uf = (

                prof[
                    'UF'
                ]

                .value_counts(
                    dropna=False
                )
            )


            for uf, qtd in (
                contagem_chunk_uf.items()
            ):

                chave = (

                    'NULO'

                    if pd.isna(
                        uf
                    )

                    else str(
                        uf
                    )
                )


                contagem_ufs[
                    chave
                ] = (

                    contagem_ufs.get(
                        chave,
                        0
                    )

                    +

                    int(
                        qtd
                    )
                )


            # ================================================
            # CONTAGEM POR FAMÍLIA
            # ================================================

            contagem_chunk_familia = (

                prof[
                    'Familia_CBO'
                ]

                .value_counts(
                    dropna=False
                )
            )


            for familia_cbo, qtd in (
                contagem_chunk_familia.items()
            ):

                chave = str(
                    familia_cbo
                )


                contagem_familias[
                    chave
                ] = (

                    contagem_familias.get(
                        chave,
                        0
                    )

                    +

                    int(
                        qtd
                    )
                )


            # ================================================
            # GARANTIR TIPOS HOMOGÊNEOS
            # ================================================

            for coluna in prof.columns:

                prof[
                    coluna
                ] = (

                    prof[
                        coluna
                    ]

                    .astype(
                        'string'
                    )
                )


            # ================================================
            # SALVAR PARQUET INCREMENTAL
            # ================================================

            tabela = (
                pa.Table.from_pandas(

                    prof,

                    preserve_index=False
                )
            )


            if writer is None:

                writer = (

                    pq.ParquetWriter(

                        saida_parquet,

                        tabela.schema,

                        compression='snappy'
                    )
                )


            writer.write_table(
                tabela
            )


            total_professores += len(
                prof
            )


            # ================================================
            # ACOMPANHAMENTO
            # ================================================

            percentual_uf_nula = (

                uf_nula_chunk

                /

                len(
                    prof
                )

                * 100
            )


            print(

                f'Chunk {numero_chunk}: '

                f'{len(chunk):,} linhas | '

                f'{len(prof):,} professores | '

                f'UF nula: '
                f'{uf_nula_chunk:,} '

                f'({percentual_uf_nula:.3f}%)'
            )


    finally:

        if writer is not None:

            writer.close()


    # ========================================================
    # RESUMO DA FUNÇÃO
    # ========================================================

    percentual_uf_nula_total = (

        total_uf_nula

        /

        total_professores

        * 100

        if total_professores > 0

        else None
    )


    return {

        'Linhas_lidas':
            total_linhas,

        'Professores':
            total_professores,

        'UF_nula':
            total_uf_nula,

        'Percentual_UF_nula':
            percentual_uf_nula_total,

        'Contagem_familias':
            contagem_familias,

        'Contagem_ufs':
            contagem_ufs
    }


# ============================================================
# 18. PROCESSAMENTO PRINCIPAL
# ============================================================

resumo = []

resumo_familias = []

resumo_ufs = []


for ano in ANOS:

    print(
        '\n'
        +
        '#' * 80
    )

    print(
        f'ANO {ano}'
    )

    print(
        '#' * 80
    )


    # --------------------------------------------------------
    # Lista FTP somente uma vez por ano
    # --------------------------------------------------------

    arquivos_ftp = listar_ftp(
        ano
    )


    # ========================================================
    # GRUPOS
    # ========================================================

    for grupo, nome_base in (
        GRUPOS.items()
    ):

        print(
            '\n'
            +
            '=' * 80
        )

        print(
            f'{ano} - {grupo}'
        )

        print(
            '=' * 80
        )


        # ----------------------------------------------------
        # Pasta do compactado
        # ----------------------------------------------------

        pasta_grupo = os.path.join(

            PASTA_BASE,

            grupo,

            str(
                ano
            )
        )


        os.makedirs(
            pasta_grupo,
            exist_ok=True
        )


        # ----------------------------------------------------
        # Pasta de saída dos professores
        # ----------------------------------------------------

        pasta_saida_ano = os.path.join(

            PASTA_FILTRADA,

            str(
                ano
            )
        )


        os.makedirs(
            pasta_saida_ano,
            exist_ok=True
        )


        # ----------------------------------------------------
        # Parquet final
        # ----------------------------------------------------

        saida_parquet = os.path.join(

            pasta_saida_ano,

            (
                f'RAIS_PROFESSORES_5CBO_'
                f'{ano}_'
                f'{grupo}.parquet'
            )
        )


        # ====================================================
        # SE PARQUET NOVO JÁ EXISTE
        # ====================================================

        if os.path.exists(
            saida_parquet
        ):

            try:

                pf = pq.ParquetFile(
                    saida_parquet
                )


                linhas_salvas = (
                    pf.metadata.num_rows
                )


                print(
                    'Parquet 5CBO '
                    'já existe.'
                )


                print(
                    f'Registros salvos: '
                    f'{linhas_salvas:,}'
                )


                resumo.append({

                    'Ano':
                        ano,

                    'Grupo':
                        grupo,

                    'Linhas_lidas':
                        None,

                    'Professores':
                        linhas_salvas,

                    'UF_nula':
                        None,

                    'Percentual_UF_nula':
                        None,

                    'Compactado_GB':
                        None,

                    'Status':
                        'JÁ PROCESSADO'
                })


                continue


            except Exception as erro:

                print(

                    'Parquet existente parece '
                    'estar corrompido.'

                )


                print(
                    f'Erro: {erro}'
                )


                print(
                    'O arquivo será removido '
                    'e reprocessado.'
                )


                os.remove(
                    saida_parquet
                )


        # ====================================================
        # PROCESSAR
        # ====================================================

        try:

            # ------------------------------------------------
            # Localizar no FTP
            # ------------------------------------------------

            arquivo_remoto = (

                localizar_arquivo(

                    arquivos_ftp,

                    nome_base
                )
            )


            nome_remoto = (

                os.path.basename(
                    arquivo_remoto
                )
            )


            print(
                f'Arquivo FTP: '
                f'{nome_remoto}'
            )


            # ------------------------------------------------
            # Caminho do .7z no Drive
            # ------------------------------------------------

            arquivo_7z = os.path.join(

                pasta_grupo,

                nome_remoto
            )


            # ------------------------------------------------
            # URL FTP
            # ------------------------------------------------

            url = (

                f'ftp://{HOST}'

                f'{BASE_FTP}/{ano}/'

                f'{quote(nome_remoto)}'
            )


            # =================================================
            # DOWNLOAD
            # =================================================

            if os.path.exists(
                arquivo_7z
            ):

                print(
                    'Arquivo compactado '
                    'já existe no Drive.'
                )


            else:

                print(
                    'Baixando...'
                )


                subprocess.run(

                    [
                        'wget',

                        '--continue',

                        '--progress=bar:force',

                        '-O',

                        arquivo_7z,

                        url
                    ],

                    check=True
                )


                print(
                    'Download concluído.'
                )


            # ------------------------------------------------
            # Tamanho compactado
            # ------------------------------------------------

            tamanho_7z = (

                os.path.getsize(
                    arquivo_7z
                )

                /

                1024**3
            )


            print(

                f'Tamanho compactado: '
                f'{tamanho_7z:.2f} GB'
            )


            # =================================================
            # PASTA TEMPORÁRIA
            # =================================================

            pasta_temp_grupo = os.path.join(

                PASTA_TEMP,

                str(
                    ano
                ),

                grupo
            )


            # ------------------------------------------------
            # Limpar sobra de execução anterior
            # ------------------------------------------------

            if os.path.exists(
                pasta_temp_grupo
            ):

                shutil.rmtree(
                    pasta_temp_grupo
                )


            os.makedirs(
                pasta_temp_grupo,
                exist_ok=True
            )


            # =================================================
            # DESCOMPACTAR
            # =================================================

            print(
                'Descompactando '
                'temporariamente no Colab...'
            )


            subprocess.run(

                [
                    '7z',

                    'x',

                    arquivo_7z,

                    f'-o{pasta_temp_grupo}',

                    '-y'
                ],

                check=True
            )


            # =================================================
            # LOCALIZAR ARQUIVOS EXTRAÍDOS
            # =================================================

            arquivos_extraidos = []


            for raiz, _, arquivos in os.walk(
                pasta_temp_grupo
            ):

                for arquivo in arquivos:

                    caminho = os.path.join(

                        raiz,

                        arquivo
                    )


                    if os.path.isfile(
                        caminho
                    ):

                        arquivos_extraidos.append(
                            caminho
                        )


            if len(
                arquivos_extraidos
            ) == 0:

                raise RuntimeError(

                    'Nenhum arquivo '
                    'foi extraído.'
                )


            print(
                'Arquivos extraídos:'
            )


            for arquivo in (
                arquivos_extraidos
            ):

                tamanho = (

                    os.path.getsize(
                        arquivo
                    )

                    /

                    1024**3
                )


                print(

                    f'  '
                    f'{os.path.basename(arquivo)} '

                    f'({tamanho:.2f} GB)'
                )


            # ------------------------------------------------
            # Segurança
            # ------------------------------------------------

            if len(
                arquivos_extraidos
            ) != 1:

                raise RuntimeError(

                    'Era esperado exatamente '
                    'um arquivo bruto dentro '
                    'do .7z.\n'

                    f'Foram encontrados: '
                    f'{arquivos_extraidos}'
                )


            arquivo_extraido = (
                arquivos_extraidos[0]
            )


            # =================================================
            # FILTRAR
            # =================================================

            print(
                '\nFiltrando professores em:'
            )


            print(
                os.path.basename(
                    arquivo_extraido
                )
            )


            resultado = (
                filtrar_professores(

                    arquivo_extraido,

                    ano,

                    grupo,

                    saida_parquet
                )
            )


            total_linhas = (
                resultado[
                    'Linhas_lidas'
                ]
            )


            total_professores = (
                resultado[
                    'Professores'
                ]
            )


            total_uf_nula = (
                resultado[
                    'UF_nula'
                ]
            )


            percentual_uf_nula = (
                resultado[
                    'Percentual_UF_nula'
                ]
            )


            # =================================================
            # VALIDAR PARQUET
            # =================================================

            if not os.path.exists(
                saida_parquet
            ):

                raise RuntimeError(

                    'Arquivo Parquet '
                    'não foi criado.'
                )


            parquet_teste = (
                pq.ParquetFile(
                    saida_parquet
                )
            )


            linhas_parquet = (

                parquet_teste
                .metadata
                .num_rows
            )


            if (

                linhas_parquet

                !=

                total_professores
            ):

                raise RuntimeError(

                    f'Divergência: '

                    f'{total_professores:,} '
                    f'professores filtrados, '

                    f'mas '

                    f'{linhas_parquet:,} '
                    f'linhas no Parquet.'
                )


            # =================================================
            # RESUMO POR FAMÍLIA
            # =================================================

            for familia_cbo, quantidade in (

                resultado[
                    'Contagem_familias'
                ]

                .items()
            ):

                resumo_familias.append({

                    'Ano':
                        ano,

                    'Grupo':
                        grupo,

                    'Familia_CBO':
                        familia_cbo,

                    'Descricao':
                        FAMILIAS_CBO.get(
                            familia_cbo
                        ),

                    'Professores':
                        quantidade
                })


            # =================================================
            # RESUMO POR UF
            # =================================================

            for uf, quantidade in (

                resultado[
                    'Contagem_ufs'
                ]

                .items()
            ):

                resumo_ufs.append({

                    'Ano':
                        ano,

                    'Grupo':
                        grupo,

                    'UF':
                        uf,

                    'Professores':
                        quantidade
                })


            # =================================================
            # MOSTRAR RESUMO
            # =================================================

            print(
                '\n'
                +
                '-' * 60
            )


            print(

                f'Total de vínculos lidos: '
                f'{total_linhas:,}'
            )


            print(

                f'Professores encontrados: '
                f'{total_professores:,}'
            )


            print(

                f'UF nula: '
                f'{total_uf_nula:,} '

                f'({percentual_uf_nula:.4f}%)'
            )


            print(
                '\nFamílias CBO:'
            )


            for familia_cbo in sorted(

                resultado[
                    'Contagem_familias'
                ]
            ):

                print(

                    f'  {familia_cbo}: '

                    f'{resultado["Contagem_familias"][familia_cbo]:,}'
                )


            print(
                '\nUFs:'
            )


            for uf in sorted(

                resultado[
                    'Contagem_ufs'
                ]
            ):

                print(

                    f'  {uf}: '

                    f'{resultado["Contagem_ufs"][uf]:,}'
                )


            print(
                '\nArquivo salvo:'
            )


            print(
                saida_parquet
            )


            # =================================================
            # APAGAR ARQUIVO BRUTO TEMPORÁRIO
            # =================================================

            shutil.rmtree(
                pasta_temp_grupo
            )


            print(
                '\nArquivo bruto temporário '
                'removido do Colab.'
            )


            # =================================================
            # RESUMO
            # =================================================

            resumo.append({

                'Ano':
                    ano,

                'Grupo':
                    grupo,

                'Linhas_lidas':
                    total_linhas,

                'Professores':
                    total_professores,

                'UF_nula':
                    total_uf_nula,

                'Percentual_UF_nula':
                    round(
                        percentual_uf_nula,
                        6
                    ),

                'Compactado_GB':
                    round(
                        tamanho_7z,
                        2
                    ),

                'Status':
                    'OK'
            })


        # ====================================================
        # TRATAMENTO DE ERRO
        # ====================================================

        except Exception as erro:

            print(
                f'ERRO: {erro}'
            )


            # ------------------------------------------------
            # Remover Parquet incompleto
            # ------------------------------------------------

            if os.path.exists(
                saida_parquet
            ):

                try:

                    os.remove(
                        saida_parquet
                    )


                    print(
                        'Parquet incompleto '
                        'foi removido.'
                    )


                except Exception:

                    pass


            # ------------------------------------------------
            # Limpar pasta temporária
            # ------------------------------------------------

            try:

                if os.path.exists(
                    pasta_temp_grupo
                ):

                    shutil.rmtree(
                        pasta_temp_grupo
                    )

            except Exception:

                pass


            resumo.append({

                'Ano':
                    ano,

                'Grupo':
                    grupo,

                'Linhas_lidas':
                    None,

                'Professores':
                    None,

                'UF_nula':
                    None,

                'Percentual_UF_nula':
                    None,

                'Compactado_GB':
                    None,

                'Status':
                    str(
                        erro
                    )
            })


# ============================================================
# 19. RESUMO GERAL
# ============================================================

df_resumo = pd.DataFrame(
    resumo
)


print(
    '\n'
    +
    '=' * 80
)

print(
    'RESUMO GERAL'
)

print(
    '=' * 80
)


display(
    df_resumo
)


# ============================================================
# 20. RESUMO POR ANO
# ============================================================

df_resumo_ano = (

    df_resumo.loc[
        df_resumo[
            'Professores'
        ]
        .notna()
    ]

    .groupby(
        'Ano',
        as_index=False
    )

    .agg(

        Professores=(
            'Professores',
            'sum'
        ),

        UF_nula=(
            'UF_nula',
            'sum'
        )
    )
)


df_resumo_ano[
    'Percentual_UF_nula'
] = (

    df_resumo_ano[
        'UF_nula'
    ]

    /

    df_resumo_ano[
        'Professores'
    ]

    * 100
)


print(
    '\n'
    +
    '=' * 80
)

print(
    'PROFESSORES POR ANO'
)

print(
    '=' * 80
)


display(
    df_resumo_ano
)


# ============================================================
# 21. RESUMO POR FAMÍLIA CBO
# ============================================================

df_familias = pd.DataFrame(
    resumo_familias
)


if len(
    df_familias
) > 0:

    df_familias_brasil = (

        df_familias

        .groupby(

            [
                'Ano',
                'Familia_CBO',
                'Descricao'
            ],

            as_index=False
        )

        [
            'Professores'
        ]

        .sum()
    )


    df_familias_brasil[
        'Percentual'
    ] = (

        df_familias_brasil[
            'Professores'
        ]

        /

        df_familias_brasil
        .groupby(
            'Ano'
        )[
            'Professores'
        ]
        .transform(
            'sum'
        )

        * 100
    )


    print(
        '\n'
        +
        '=' * 80
    )

    print(
        'PROFESSORES POR FAMÍLIA CBO'
    )

    print(
        '=' * 80
    )


    display(
        df_familias_brasil
    )


# ============================================================
# 22. RESUMO POR UF
# ============================================================

df_ufs = pd.DataFrame(
    resumo_ufs
)


if len(
    df_ufs
) > 0:

    df_ufs_brasil = (

        df_ufs

        .loc[
            df_ufs[
                'UF'
            ]
            !=
            'NULO'
        ]

        .groupby(

            [
                'Ano',
                'UF'
            ],

            as_index=False
        )

        [
            'Professores'
        ]

        .sum()
    )


    df_ufs_brasil[
        'Percentual_Brasil'
    ] = (

        df_ufs_brasil[
            'Professores'
        ]

        /

        df_ufs_brasil
        .groupby(
            'Ano'
        )[
            'Professores'
        ]
        .transform(
            'sum'
        )

        * 100
    )


    print(
        '\n'
        +
        '=' * 80
    )

    print(
        'PROFESSORES POR UF'
    )

    print(
        '=' * 80
    )


    display(
        df_ufs_brasil
    )


# ============================================================
# 23. SALVAR RESUMOS
# ============================================================

PASTA_RESUMOS = os.path.join(

    PASTA_FILTRADA,

    '_RESUMOS'
)


os.makedirs(
    PASTA_RESUMOS,
    exist_ok=True
)


# ------------------------------------------------------------
# Resumo geral
# ------------------------------------------------------------

df_resumo.to_csv(

    os.path.join(
        PASTA_RESUMOS,
        '01_resumo_processamento.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ------------------------------------------------------------
# Resumo por ano
# ------------------------------------------------------------

df_resumo_ano.to_csv(

    os.path.join(
        PASTA_RESUMOS,
        '02_professores_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ------------------------------------------------------------
# Famílias
# ------------------------------------------------------------

if len(
    df_familias
) > 0:

    df_familias.to_csv(

        os.path.join(
            PASTA_RESUMOS,
            '03_familias_por_grupo.csv'
        ),

        index=False,

        encoding='utf-8-sig'
    )


    df_familias_brasil.to_csv(

        os.path.join(
            PASTA_RESUMOS,
            '04_familias_brasil.csv'
        ),

        index=False,

        encoding='utf-8-sig'
    )


# ------------------------------------------------------------
# UFs
# ------------------------------------------------------------

if len(
    df_ufs
) > 0:

    df_ufs.to_csv(

        os.path.join(
            PASTA_RESUMOS,
            '05_ufs_por_grupo.csv'
        ),

        index=False,

        encoding='utf-8-sig'
    )


    df_ufs_brasil.to_csv(

        os.path.join(
            PASTA_RESUMOS,
            '06_professores_por_uf.csv'
        ),

        index=False,

        encoding='utf-8-sig'
    )


# ============================================================
# 24. VALIDAÇÕES FINAIS
# ============================================================

print(
    '\n'
    +
    '=' * 80
)

print(
    'VALIDAÇÕES FINAIS'
)

print(
    '=' * 80
)


# ------------------------------------------------------------
# Número de arquivos OK
# ------------------------------------------------------------

quantidade_ok = (

    df_resumo[
        'Status'
    ]

    .isin(
        [
            'OK',
            'JÁ PROCESSADO'
        ]
    )

    .sum()
)


print(

    f'Arquivos processados/validados: '
    f'{quantidade_ok} de 36'
)


# ------------------------------------------------------------
# Erros
# ------------------------------------------------------------

df_erros = (

    df_resumo.loc[

        ~df_resumo[
            'Status'
        ]

        .isin(
            [
                'OK',
                'JÁ PROCESSADO'
            ]
        )
    ]
)


if len(
    df_erros
) == 0:

    print(
        'Nenhum erro de processamento.'
    )

else:

    print(
        '\nATENÇÃO: existem erros:'
    )

    display(
        df_erros
    )


# ------------------------------------------------------------
# UFs nulas
# ------------------------------------------------------------

if (
    'Percentual_UF_nula'
    in df_resumo.columns
):

    df_nulos = (

        df_resumo.loc[

            df_resumo[
                'Percentual_UF_nula'
            ]

            .fillna(0)

            >
            1
        ]
    )


    if len(
        df_nulos
    ) == 0:

        print(
            'Nenhum arquivo apresentou '
            'mais de 1% de UF nula.'
        )

    else:

        print(
            '\nATENÇÃO: arquivos com '
            'mais de 1% de UF nula:'
        )

        display(
            df_nulos[
                [
                    'Ano',
                    'Grupo',
                    'Professores',
                    'UF_nula',
                    'Percentual_UF_nula'
                ]
            ]
        )


print(
    '\nResultados salvos em:'
)

print(
    PASTA_FILTRADA
)

print(
    '\nResumos salvos em:'
)

print(
    PASTA_RESUMOS
)